In [ ]:
import json
import os

from typing import Any, Dict, Iterable, List, Optional

import geopandas as gpd
import pandas as pd
import shapely.geometry as sg

In [ ]:
def yolo_string_to_dict(yolo_annotation: str) -> Dict[str, Any]:
    """
    Convert a line of YOLO annotation text to a dict format.

    Expects `cat_id x_center y_center width height conf`.

    Returns `{"category": int, "confidence": float, "geometry": shapely.geometry.box}`.
    """
    values = yolo_annotation.split()

    if len(values) >= 6:
        cat_id, x_center, y_center, width, height, conf = map(float, values[:6])
    elif len(values) == 5:
        cat_id, x_center, y_center, width, height = map(float, values)
        conf = 1.0
    else:
        raise ValueError(f"Unrecognized annotation string format: {yolo_annotation}")

    bbox = sg.box(
        minx=x_center - width / 2,
        miny=y_center - height / 2,
        maxx=x_center + width / 2,
        maxy=y_center + height / 2,
    )

    data = {"category": int(cat_id), "confidence": conf, "geometry": bbox}

    return data

def yolo_file_to_dicts(annotation_file_path: str) -> List[Dict[str, Any]]:
    """
    Read a YOLO annotations file and return the annotations as a list of dicts.

    See `yolo_string_to_dict(..)` for details on the dict contents.
    """
    data = []

    with open(annotation_file_path, "r") as f:
        file_name = os.path.basename(annotation_file_path)
        for line in f.readlines():
            line_data = yolo_string_to_dict(line)
            line_data["file_name"] = file_name
            data.append(line_data)

    return data

def read_annotations_folder(
    folder_path: str, categories: Optional[Iterable[int]], agnostic: bool = False
) -> gpd.GeoDataFrame:
    """
    Convert all YOLO annotation files in a folder to GeoDataFrame with one annotation per row.

    The GeoDataFrame has columns `"file_name", "category", "confidence", "geometry"`.
    """
    data = []
    annotation_files = [
        file for file in os.listdir(folder_path) if os.path.splitext(file)[1] == ".txt"
    ]

    for file in annotation_files:
        data.extend(yolo_file_to_dicts(os.path.join(folder_path, file)))

    gdf = gpd.GeoDataFrame(
        data=data, columns=["file_name", "category", "confidence", "geometry"]
    )

    if categories is not None:
        gdf = gdf[gdf["category"].isin(categories)]

    if agnostic:
        gdf["category"] == 0

    return gdf

def read_coco_annotations(json_file: str, categories: Optional[Iterable[int]]) -> gpd.GeoDataFrame:
    with open(json_file) as f:
        json_content = json.load(f)

    images_df = pd.DataFrame(json_content["images"]).set_index("id")
    images_df["name"] = [
        os.path.splitext(os.path.basename(file))[0]
        for file in images_df["file_name"]
    ]
    images_df = images_df[["name"]]

    annotations_df = pd.DataFrame(json_content["annotations"]).set_index("id")
    annotations_df["category_id"] = annotations_df["category_id"] - 1
    annotations_df = annotations_df[annotations_df["category_id"].isin(categories)]

    return annotations_df.join(images_df, on="image_id", how="left")

In [ ]:
dataset_folder = "../datasets/oor/testride_velotech/"

images_folder = os.path.join(dataset_folder, "recording_2025-05-14_19-47-40/images")
detections_folder = os.path.join(dataset_folder, "recording_2025-05-14_19-47-40_all/labels")
annotations_file = os.path.join(dataset_folder, "recording_2025-05-14_19-47-40_annotated.json")
cluster_annotations_file = os.path.join(dataset_folder, "recording_2025-05-14_reviewed.csv")

categories = {
    2: "Container",
    3: "Dixie",
}

In [ ]:
clusters_df = pd.read_csv(cluster_annotations_file)
clusters_df = clusters_df[clusters_df["object_category"].isin(categories.keys())]
clusters_df["name"] = [
    os.path.splitext(file)[0]
    for file in clusters_df["image_name"]
]
clusters_df.sort_values(by="name", ascending=True, inplace=True)

In [ ]:
annotations_df = read_coco_annotations(json_file=annotations_file, categories=categories.keys())

In [ ]:
detections_df = read_annotations_folder(detections_folder, categories=categories.keys())
detections_df["name"] = [
    os.path.splitext(file)[0]
    for file in detections_df["file_name"]
]
detections_df.sort_values(by="name", ascending=True, inplace=True)

In [ ]:
clusters_names = set(clusters_df["name"].unique())
annotations_names = set(annotations_df["name"].unique())
detections_names = set(detections_df["name"].unique())

In [ ]:
# Check if annotations and clusters contain the same set of photos
clusters_names == annotations_names

In [ ]:
# Copy photos into folders per cluster to check
import shutil

clusters_folder = os.path.join(dataset_folder, "clusters")

for row in clusters_df.itertuples():
    image_path = os.path.join(images_folder, row.image_name)
    out_folder = os.path.join(clusters_folder, str(row.ID))
    os.makedirs(out_folder, exist_ok=True)
    shutil.copy2(image_path, out_folder)